# 07: BitNet demo

[microsoft/bitnet-b1.58-2B-4T](https://huggingface.co/microsoft/bitnet-b1.58-2B-4T)
(30 layers, dim=2560, 2B params). Weights stored as packed uint8 with per-layer weight_scale.

smelt.quantize() detects the ternary weights, reads weight_scale, packs into TL1 for AVX2 inference.

In [1]:
import pathlib
import sys
import time

sys.path.insert(0, str(pathlib.Path("../")))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

## smelt.quantize

In [2]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
w0 = model.model.layers[0].self_attn.q_proj.weight
print(f"params: {n_params:.0f}M")
print(f"weight dtype: {w0.dtype}")
print(f"weight unique values: {sorted(w0.unique().tolist())}")
print(f"has weight_scale: {hasattr(model.model.layers[0].self_attn.q_proj, 'weight_scale')}")

You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

params: 2413M
weight dtype: torch.uint8
weight unique values: [0, 1, 255]
has weight_scale: True


In [3]:
t0 = time.perf_counter()
smelt.quantize(model)
print(f"quantized in {time.perf_counter() - t0:.1f}s")

quantized in 10.4s


## Generate

In [4]:
prompts = [
    "The capital of UK is",
    "def fibonacci(n):",
    "Explain x86 AVX2 in simple terms:",
]

for prompt in prompts:
    ids = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=64, do_sample=False)

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"--- {prompt} ---")
    print(text)
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your 

--- The capital of UK is ---
The capital of UK is London. London is a city in the United Kingdom. It is the largest city in the UK and is located in the south east of England. London is a major financial and cultural center of the world. It is also the largest city in the UK by population. London is a city with a rich history and is home



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- def fibonacci(n): ---
def fibonacci(n): 
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

def is_prime(n):
    if n <= 1:
        return False
    for i in range(2

--- Explain x86 AVX2 in simple terms: ---
Explain x86 AVX2 in simple terms: x86 AVX2 is a technology that helps computers do things faster. It's like having a superpower that makes the computer work better and faster. It's used to make programs run faster and to do more things at the same time. It's like having a supercomputer inside your computer that helps it do its



## Benchmark

64 tokens, 3 runs avg.

In [5]:
import platform
import subprocess


def get_cpu_name():
    try:
        out = subprocess.check_output(["lscpu"], text=True)
        for line in out.splitlines():
            if "Model name" in line:
                return line.split(":")[1].strip()

    except Exception:
        pass

    return platform.processor() or platform.machine()


def bench_generate(model, input_ids, n_tokens=64, warmup=2, runs=3):
    for _ in range(warmup):
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=n_tokens, do_sample=False)

    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(input_ids, max_new_tokens=n_tokens, do_sample=False)
        times.append(time.perf_counter() - t0)

    return n_tokens / (sum(times) / len(times))


prompt_ids = tokenizer.encode("The meaning of life is", return_tensors="pt")
cpu_name = get_cpu_name()

tps = bench_generate(model, prompt_ids)
print(f"smelt ({cpu_name}): {tps:.1f} tok/s")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

smelt (Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz): 5.7 tok/s
